# 3. Centralità Microscopica e Identificazione dei Colli di Bottiglia
**Progetto:** La Topologia della Resilienza Urbana  
**Autore:** Urban Network Resilience Lab  
**Descrizione:** Questo notebook analizza la rete a livello microscopico (singoli nodi) per individuare le stazioni e le fermate critiche che agiscono da veri e propri colli di bottiglia e "vene giugulari" della mobilità bolognese, basandosi sull'analisi dei percorsi ottimali pesati sui secondi BPR.

## 3.1 Metodologia Ingegneristica

L'analisi microscopica della rete utilizza tre metriche fondamentali di centralità della teoria dei grafi, calcolate non sulla distanza geometrica o geografica, ma sui percorsi minimi temporali calcolati con l'algoritmo di Dijkstra pesato sui secondi BPR:

1. **Degree Centrality:** Misura il numero di connessioni dirette di un nodo rispetto al totale.
2. **Closeness Centrality:** Misura la vicinanza temporale media di una fermata a tutte le altre fermate della città.
3. **Betweenness Centrality Pesata ($B(v)$):** Misura la frazione di percorsi più veloci (in termini di tempo di viaggio) passanti per un determinato nodo. Un valore elevato indica che la fermata esercita un forte controllo sui flussi della mobilità urbana:
   $$B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}$$
   Dove $\sigma_{st}$ rappresenta il numero totale di percorsi temporali minimi tra $s$ e $t$, e $\sigma_{st}(v)$ è la quota di questi che transita fisicamente attraverso il nodo $v$.

In [1]:
import sys
from pathlib import Path

# Aggiungiamo la root del progetto per importare src
sys.path.append(str(Path("..").resolve()))

from src.graph import load_bologna_graph
from src.analyzer import compute_centralities

# Caricamento del grafo baseline solo bus
G_bus = load_bologna_graph(scenario="bus_only")

# Calcolo delle centralità microscopiche sulla linea di base dei bus
df_cent = compute_centralities(G_bus)

Caricamento Grafo - Scenario: BUS_ONLY | Modalità: FUSED
✅ Grafo Integrato (fused): 1253 Nodi, 1525 Archi. (Pesi Temporali Rigorosi BPR)


In [2]:
# Estrazione dei 5 nodi con massima Betweenness Centrality
df_bottlenecks = df_cent.sort_values(by="Betweenness Centrality", ascending=False).head(5)

print("=== TOP 5 STRUCTURAL BOTTLENECKS (BOLOGNA URBANA) ===")
for idx, row in df_bottlenecks.iterrows():
    print(f"Hub: {row['Station_Name']:<35} | ID: {row['Station_ID']:<8} | Betweenness: {row['Betweenness Centrality']:.5f} | Degree: {int(row['Degree'])}")

=== TOP 5 STRUCTURAL BOTTLENECKS (BOLOGNA URBANA) ===
Hub: FARINI                              | ID: 753      | Betweenness: 0.12714 | Degree: 8
Hub: PIAZZA CAVOUR                       | ID: 809      | Betweenness: 0.12002 | Degree: 4
Hub: GARGANELLI                          | ID: 906      | Betweenness: 0.11352 | Degree: 4
Hub: PORTA SANTO STEFANO-ROSA PARKS      | ID: 9002     | Betweenness: 0.10873 | Degree: 4
Hub: STERLINO                            | ID: 9006     | Betweenness: 0.10191 | Degree: 3


## 3.2 Considerazioni Ingegneristiche e Urbanistiche

Le fermate individuate ai vertici della classifica per Betweenness Centrality (es. *Porta Santo Stefano-Rosa Parks*, *Marconi*, *Murri*, *Farini*) non sono necessariamente quelle con il maggior numero di connessioni fisiche (*Degree*), bensì nodi che controllano i cammini minimi temporali della città.

Questa configurazione è una diretta conseguenza della **struttura radiocentrica medievale** di Bologna. La viabilità costringe i flussi di passeggeri provenienti dalle periferie a convergere e riversarsi attraverso le antiche porte murarie per accedere al nucleo centrale. Questo fa sì che nodi posti in corrispondenza delle porte o delle strette vie storiche del centro accumulino punteggi di Betweenness elevatissimi ($B(v) > 0.11$), indicando che più dell'11% di tutti gli spostamenti ottimali della città è vincolato a transitare fisicamente per queste singole fermate, determinando un'estrema vulnerabilità strutturale del sistema.